# Notebook 12 — ASL Top-50 Inference

This notebook has two separate run paths:

## Mode A — Single video test
Use this for one user-uploaded clip. Optional expected-label dropdown gives `Correct: True/False`. Results append to:

`/content/drive/MyDrive/asl/notebook12_user_video_results.jsonl`

## Mode B — Batch WLASL Top-50 test
Use this for a zip of many WLASL Top-50 videos plus optional `labels.csv`. Results append to:

`/content/drive/MyDrive/asl/notebook12_wlasl_top50_batch_results.jsonl`

Run **Common setup** first, then choose either Mode A or Mode B. Load the model once per runtime.


## Common setup

Run these cells once at the top of the runtime. They install dependencies, load helper functions, mount Drive, configure paths, and load the canonical Top-50 label list.


In [ ]:
# Runtime setup (Colab)
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")

!pip -q install opencv-python-headless huggingface_hub ipywidgets
!pip -q install --upgrade --no-cache-dir unsloth


In [ ]:
# Self-contained Notebook 12 helpers (no repo clone/import required)
from __future__ import annotations

import csv
import json
import re
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal, Sequence

NOTEBOOK12_MODEL_REPO = "AlexD281/asl-gemma4-26b-a4b-zahid-pretrain-lora"
NOTEBOOK12_BASE_MODEL = "unsloth/gemma-4-26B-A4B-it"
NOTEBOOK12_NUM_FRAMES = 30
NOTEBOOK12_FRAME_SIZE = 448
NOTEBOOK12_MAX_CLIP_SECONDS = 5.0

PredictionStatus = Literal["ok", "uncertain"]


@dataclass(frozen=True)
class Notebook12Config:
    model_repo: str = NOTEBOOK12_MODEL_REPO
    base_model: str = NOTEBOOK12_BASE_MODEL
    num_frames: int = NOTEBOOK12_NUM_FRAMES
    frame_size: int = NOTEBOOK12_FRAME_SIZE
    max_clip_seconds: float = NOTEBOOK12_MAX_CLIP_SECONDS


@dataclass(frozen=True)
class StrictPrediction:
    status: PredictionStatus
    predicted_gloss: str | None
    raw_output: str


class DurationLimitError(ValueError):
    def __init__(self, duration_seconds: float, max_seconds: float) -> None:
        super().__init__(f"Video is {duration_seconds:.2f}s; maximum allowed is {max_seconds:.2f}s.")
        self.duration_seconds = duration_seconds
        self.max_seconds = max_seconds


DEFAULT_NOTEBOOK12_CONFIG = Notebook12Config()


def normalize_gloss(text: str) -> str:
    normalized = str(text).strip().lower()
    normalized = re.sub(r"^`+|`+$", "", normalized)
    normalized = re.sub(r"[^a-z0-9_ -]+", "", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized


def load_top50_labels(path: str | Path) -> tuple[str, ...]:
    labels = [normalize_gloss(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]
    labels = [label for label in labels if label]
    if len(labels) != 50:
        raise ValueError(f"Expected exactly 50 Top-50 labels, found {len(labels)} in {path}.")
    if len(set(labels)) != 50:
        raise ValueError(f"Top-50 labels must be unique in {path}.")
    return tuple(labels)


def strict_top50_prediction(raw_output: str, labels: Sequence[str]) -> StrictPrediction:
    normalized_labels = {normalize_gloss(label) for label in labels}
    first_line = str(raw_output).splitlines()[0] if str(raw_output).splitlines() else ""
    candidate = normalize_gloss(first_line)
    if candidate in normalized_labels:
        return StrictPrediction(status="ok", predicted_gloss=candidate, raw_output=raw_output)
    return StrictPrediction(status="uncertain", predicted_gloss=None, raw_output=raw_output)


def build_top50_prompt(labels: Sequence[str]) -> str:
    normalized_labels = [normalize_gloss(label) for label in labels]
    if not normalized_labels:
        raise ValueError("labels must not be empty.")
    return (
        "Identify the ASL sign shown across these frames. "
        "Return exactly one gloss label from the approved list.\n"
        f"Approved labels: {', '.join(normalized_labels)}"
    )


def deterministic_translation(gloss: str | None) -> str | None:
    if gloss is None:
        return None
    normalized = normalize_gloss(gloss)
    if not normalized:
        return None
    return normalized[0].upper() + normalized[1:] + "."


def validate_clip_duration(duration_seconds: float, *, max_seconds: float = NOTEBOOK12_MAX_CLIP_SECONDS) -> None:
    if duration_seconds > max_seconds:
        raise DurationLimitError(duration_seconds=duration_seconds, max_seconds=max_seconds)


def even_sample_indices(*, source_count: int, target_count: int = NOTEBOOK12_NUM_FRAMES) -> tuple[int, ...]:
    if target_count <= 0:
        raise ValueError("target_count must be positive.")
    if source_count < target_count:
        raise ValueError(f"Cannot sample {target_count} frames from only {source_count} decoded frames.")
    if target_count == 1:
        return (0,)
    return tuple(round(i * (source_count - 1) / (target_count - 1)) for i in range(target_count))


def probe_video_duration_seconds(video_path: str | Path) -> float:
    try:
        import cv2  # type: ignore
    except ImportError as exc:
        raise RuntimeError("OpenCV (cv2) is required to probe uploaded videos.") from exc

    capture = cv2.VideoCapture(str(video_path))
    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frame_count = float(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0.0)
    finally:
        capture.release()
    if fps <= 0 or frame_count <= 0:
        raise ValueError(f"Could not determine duration for video: {video_path}")
    return frame_count / fps


def extract_exact_frame_images(
    video_path: str | Path,
    output_dir: str | Path,
    *,
    num_frames: int = NOTEBOOK12_NUM_FRAMES,
    frame_size: int = NOTEBOOK12_FRAME_SIZE,
) -> list[Path]:
    try:
        import cv2  # type: ignore
    except ImportError as exc:
        raise RuntimeError("OpenCV (cv2) is required to extract uploaded video frames.") from exc

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)

    capture = cv2.VideoCapture(str(video_path))
    frames: list[Any] = []
    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            frames.append(frame)
    finally:
        capture.release()

    indices = even_sample_indices(source_count=len(frames), target_count=num_frames)
    written: list[Path] = []
    for out_index, source_index in enumerate(indices):
        frame = frames[source_index]
        resized = cv2.resize(frame, (frame_size, frame_size), interpolation=cv2.INTER_AREA)
        out_path = output / f"frame_{out_index:03d}.jpg"
        if not cv2.imwrite(str(out_path), resized):
            raise RuntimeError(f"Failed to write extracted frame: {out_path}")
        written.append(out_path)
    return written


def build_result_row(
    *,
    video_filename: str,
    duration_seconds: float,
    expected_label: str | None,
    prediction: StrictPrediction,
    config: Notebook12Config = DEFAULT_NOTEBOOK12_CONFIG,
    timestamp: str | None = None,
) -> dict[str, Any]:
    expected = normalize_gloss(expected_label) if expected_label else None
    predicted = prediction.predicted_gloss
    return {
        "timestamp": timestamp or datetime.now(timezone.utc).isoformat(),
        "video_filename": video_filename,
        "duration_seconds": duration_seconds,
        "expected_label": expected,
        "status": prediction.status,
        "predicted_gloss": predicted,
        "translation": deterministic_translation(predicted),
        "raw_output": prediction.raw_output,
        "correct": (predicted == expected) if expected else None,
        "model": config.model_repo,
        "base_model": config.base_model,
        "num_frames": config.num_frames,
        "frame_size": config.frame_size,
    }


def append_result_jsonl(path: str | Path, row: dict[str, Any]) -> None:
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True))
        handle.write("\n")


def load_batch_expected_labels(path: str | Path) -> dict[str, str]:
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"filename", "expected_label"}
        if not required.issubset(set(reader.fieldnames or [])):
            raise ValueError("labels.csv must contain filename and expected_label columns.")
        labels: dict[str, str] = {}
        for row in reader:
            filename = Path(str(row.get("filename") or "").strip()).name
            expected = normalize_gloss(str(row.get("expected_label") or ""))
            if filename and expected:
                labels[filename] = expected
    return labels


def list_video_files(root: str | Path) -> tuple[Path, ...]:
    video_root = Path(root)
    suffixes = {".mp4", ".mov", ".m4v"}
    return tuple(
        sorted(
            (path for path in video_root.rglob("*") if path.is_file() and path.suffix.lower() in suffixes),
            key=lambda path: path.name.lower(),
        )
    )


def build_batch_summary(rows: Sequence[dict[str, Any]]) -> dict[str, Any]:
    total = len(rows)
    labeled = sum(1 for row in rows if row.get("expected_label"))
    ok = sum(1 for row in rows if row.get("status") == "ok")
    correct = sum(1 for row in rows if row.get("correct") is True)
    uncertain = sum(1 for row in rows if row.get("status") == "uncertain")
    failed = sum(1 for row in rows if row.get("status") == "error")
    incorrect = sum(1 for row in rows if row.get("expected_label") and row.get("correct") is False)
    accuracy = (correct / labeled) if labeled else None
    return {
        "total": total,
        "labeled": labeled,
        "ok": ok,
        "correct": correct,
        "incorrect": incorrect,
        "uncertain": uncertain,
        "failed": failed,
        "accuracy_on_labeled": accuracy,
    }


def config_as_dict(config: Notebook12Config = DEFAULT_NOTEBOOK12_CONFIG) -> dict[str, Any]:
    return asdict(config)

print("Notebook 12 helpers loaded (self-contained)")
print("Config:", DEFAULT_NOTEBOOK12_CONFIG)


In [ ]:
# Mount Drive for labels + append-only results log
from google.colab import drive

drive.mount('/content/drive')
print("Drive mounted")


In [ ]:
# Directly editable config
from pathlib import Path

HF_NAMESPACE = "AlexD281"
BASE_MODEL = "unsloth/gemma-4-26B-A4B-it"
STAGE1_RUN_NAME = "asl-gemma4-26b-a4b-zahid-pretrain-lora"
ADAPTER_REPO = f"{HF_NAMESPACE}/{STAGE1_RUN_NAME}"

MAX_CLIP_SECONDS = 5.0
NUM_FRAMES = 30
FRAME_SIZE = 448
MAX_SEQ_LENGTH = 8192

TOP50_LABELS_FILE = Path("/content/drive/MyDrive/asl/video_finetune/top50/labels.txt")
RESULTS_JSONL = Path("/content/drive/MyDrive/asl/notebook12_user_video_results.jsonl")
WORK_DIR = Path("/content/asl_notebook12_user_video")
FRAME_DIR = WORK_DIR / "frames"
WORK_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

print("Base model:", BASE_MODEL)
print("Adapter repo:", ADAPTER_REPO)
print("Labels file:", TOP50_LABELS_FILE)
print("Results log:", RESULTS_JSONL)


In [ ]:
# Load and verify canonical Top-50 allowlist
labels = load_top50_labels(TOP50_LABELS_FILE)
print("Top-50 labels loaded:", len(labels))
print(labels)


## Mode A — Single video test

Run this section when you want to test exactly one uploaded video. Choose an expected label before logging if you want `Correct` to be `True` or `False`; leave it blank for demo mode.


In [ ]:
# Mode A setup: optional expected-label dropdown
import ipywidgets as widgets
from IPython.display import display

expected_label_dropdown = widgets.Dropdown(
    options=[""] + sorted(labels),
    value="",
    description="Expected:",
    disabled=False,
)
display(expected_label_dropdown)
print("Leave blank for demo mode, or choose a Top-50 label for correctness logging.")


In [ ]:
# Mode A: upload exactly one user video
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Upload exactly one video file; got {len(uploaded)} files.")

video_filename = next(iter(uploaded.keys()))
video_path = Path(video_filename).resolve()
if video_path.suffix.lower() not in {".mp4", ".mov", ".m4v"}:
    raise ValueError(f"Expected .mp4/.mov/.m4v video, got: {video_path.name}")

duration_seconds = probe_video_duration_seconds(video_path)
validate_clip_duration(duration_seconds, max_seconds=MAX_CLIP_SECONDS)

print(f"Uploaded: {video_path.name}")
print(f"Duration: {duration_seconds:.2f}s / max {MAX_CLIP_SECONDS:.2f}s")


In [ ]:
# Mode A: extract exactly 30 evenly sampled 448x448 frames
import shutil

if FRAME_DIR.exists():
    shutil.rmtree(FRAME_DIR)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

frame_paths = extract_exact_frame_images(
    video_path,
    FRAME_DIR,
    num_frames=NUM_FRAMES,
    frame_size=FRAME_SIZE,
)
assert len(frame_paths) == NUM_FRAMES, f"Expected {NUM_FRAMES} frames, got {len(frame_paths)}"
print("Extracted frames:", len(frame_paths))
print("First frame:", frame_paths[0])
print("Last frame:", frame_paths[-1])


## Shared model load

Run this once before either prediction path. After the model is loaded, you can use Mode A single-video inference or Mode B batch inference without reloading.


In [ ]:
# Load Notebook 11 model from Hugging Face
import gc
import os

import torch
from huggingface_hub import login, snapshot_download
from unsloth import FastVisionModel

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU runtime is required for Notebook 11 26B inference.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} | VRAM: {gpu_mem_gb:.1f} GB")
print("Loading adapter snapshot:", ADAPTER_REPO)

gc.collect()
torch.cuda.empty_cache()
adapter_local = snapshot_download(ADAPTER_REPO, token=hf_token)
print("Adapter local path:", adapter_local)

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=adapter_local,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    max_seq_length=MAX_SEQ_LENGTH,
    device_map={"": 0},
)
FastVisionModel.for_inference(model)
print("Model ready")


In [ ]:
# Mode A: run constrained Top-50 prediction
from PIL import Image

PROMPT = build_top50_prompt(labels)

content = [{"type": "text", "text": PROMPT}]
for frame_path in frame_paths:
    content.append({"type": "image", "image": Image.open(frame_path).convert("RGB")})

user_msg = {"role": "user", "content": content}
inputs = tokenizer.apply_chat_template(
    [user_msg],
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
if hasattr(inputs, "to"):
    inputs = inputs.to(model.device)
else:
    inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}

prompt_len = int(inputs["input_ids"].shape[-1])
out = model.generate(**inputs, max_new_tokens=12, do_sample=False)
raw_output = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
prediction = strict_top50_prediction(raw_output, labels)

print("Raw model output:", repr(raw_output))
if prediction.status == "ok":
    print(f"Top-50 guess: {prediction.predicted_gloss}")
else:
    print("Top-50 guess: uncertain")
print("Warning: this notebook only recognizes the current Top-50 label set.")


In [ ]:
# Mode A: display translation and append single-video JSONL row
expected_label = expected_label_dropdown.value or None
row = build_result_row(
    video_filename=video_path.name,
    duration_seconds=duration_seconds,
    expected_label=expected_label,
    prediction=prediction,
)
append_result_jsonl(RESULTS_JSONL, row)

print("Status:", row["status"])
print("Predicted gloss:", row["predicted_gloss"] or "uncertain")
print("Translation:", row["translation"] or "Uncertain.")
print("Expected label:", row["expected_label"])
print("Correct:", row["correct"])
print("Logged metadata/results only to:", RESULTS_JSONL)
print(row)

# Privacy cleanup: remove the raw upload from the Colab runtime after logging.
# Extracted frames remain under /content for debugging this run; delete FRAME_DIR manually if desired.
if video_path.exists():
    video_path.unlink()
    print("Removed raw uploaded video from runtime:", video_path.name)


## Mode B — Batch WLASL Top-50 test

Run this section after the shared model-load cell when you want to score many WLASL Top-50 videos. Upload a zip containing `.mp4`/`.mov`/`.m4v` files and an optional `labels.csv` with columns `filename,expected_label`. Videos may be under a `videos/` folder or nested directories.


In [ ]:
# Mode B: upload and unpack a WLASL Top-50 batch zip
from google.colab import files
import shutil
import zipfile

BATCH_ROOT = WORK_DIR / "batch"
BATCH_EXTRACT_DIR = BATCH_ROOT / "extracted"
BATCH_FRAME_ROOT = BATCH_ROOT / "frames"
BATCH_RESULTS_JSONL = Path("/content/drive/MyDrive/asl/notebook12_wlasl_top50_batch_results.jsonl")

uploaded_batch = files.upload()
if len(uploaded_batch) != 1:
    raise ValueError(f"Upload exactly one batch .zip; got {len(uploaded_batch)} files.")

batch_zip_name = next(iter(uploaded_batch.keys()))
batch_zip_path = Path(batch_zip_name).resolve()
if batch_zip_path.suffix.lower() != ".zip":
    raise ValueError(f"Expected a .zip file, got: {batch_zip_path.name}")

if BATCH_ROOT.exists():
    shutil.rmtree(BATCH_ROOT)
BATCH_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_FRAME_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(batch_zip_path) as zf:
    zf.extractall(BATCH_EXTRACT_DIR)

batch_video_paths = list_video_files(BATCH_EXTRACT_DIR)
if not batch_video_paths:
    raise RuntimeError("No .mp4/.mov/.m4v videos found in the batch zip.")

label_csv_candidates = sorted(BATCH_EXTRACT_DIR.rglob("labels.csv"))
batch_expected_labels = load_batch_expected_labels(label_csv_candidates[0]) if label_csv_candidates else {}

print("Batch zip:", batch_zip_path.name)
print("Videos found:", len(batch_video_paths))
print("labels.csv:", label_csv_candidates[0] if label_csv_candidates else "not provided")
print("Labeled videos:", len(batch_expected_labels))
print("Batch results log:", BATCH_RESULTS_JSONL)


In [ ]:
# Mode B: run batch inference and append one JSONL row per video
from PIL import Image


def predict_uploaded_video(frame_paths_for_video):
    prompt = build_top50_prompt(labels)
    content = [{"type": "text", "text": prompt}]
    for frame_path in frame_paths_for_video:
        content.append({"type": "image", "image": Image.open(frame_path).convert("RGB")})

    user_msg = {"role": "user", "content": content}
    inputs = tokenizer.apply_chat_template(
        [user_msg],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    if hasattr(inputs, "to"):
        inputs = inputs.to(model.device)
    else:
        inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}

    prompt_len = int(inputs["input_ids"].shape[-1])
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False)
    raw = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return strict_top50_prediction(raw, labels)

batch_rows = []
for index, batch_video_path in enumerate(batch_video_paths, start=1):
    print(f"[{index}/{len(batch_video_paths)}] {batch_video_path.name}")
    frame_dir = BATCH_FRAME_ROOT / batch_video_path.stem
    try:
        duration = probe_video_duration_seconds(batch_video_path)
        validate_clip_duration(duration, max_seconds=MAX_CLIP_SECONDS)
        if frame_dir.exists():
            shutil.rmtree(frame_dir)
        frame_dir.mkdir(parents=True, exist_ok=True)
        batch_frame_paths = extract_exact_frame_images(
            batch_video_path,
            frame_dir,
            num_frames=NUM_FRAMES,
            frame_size=FRAME_SIZE,
        )
        batch_prediction = predict_uploaded_video(batch_frame_paths)
        expected = batch_expected_labels.get(batch_video_path.name)
        row = build_result_row(
            video_filename=batch_video_path.name,
            duration_seconds=duration,
            expected_label=expected,
            prediction=batch_prediction,
        )
    except Exception as exc:
        expected = batch_expected_labels.get(batch_video_path.name)
        row = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "video_filename": batch_video_path.name,
            "duration_seconds": None,
            "expected_label": expected,
            "status": "error",
            "predicted_gloss": None,
            "translation": None,
            "raw_output": None,
            "correct": None,
            "error": f"{type(exc).__name__}: {exc}",
            "model": ADAPTER_REPO,
            "base_model": BASE_MODEL,
            "num_frames": NUM_FRAMES,
            "frame_size": FRAME_SIZE,
        }
        print("  ERROR:", row["error"])

    append_result_jsonl(BATCH_RESULTS_JSONL, row)
    batch_rows.append(row)
    print("  status=", row["status"], "pred=", row["predicted_gloss"], "expected=", row["expected_label"], "correct=", row["correct"])

batch_summary = build_batch_summary(batch_rows)
print("\nBatch summary")
print(batch_summary)
print("Logged batch results to:", BATCH_RESULTS_JSONL)

# Privacy cleanup: remove uploaded zip from runtime after logging.
if batch_zip_path.exists():
    batch_zip_path.unlink()
    print("Removed raw batch zip from runtime:", batch_zip_path.name)


## MVP verification gate

Notebook 12 is done when 3 known Top-50 uploads run end-to-end, all 3 append JSONL rows, at least 1 prediction is correct, invalid/uncertain behavior is non-crashing, and >5s videos are blocked with a clear error.
